In [1]:
class PrangeISD():
    
    def __init__(self, H, syndrome, t):
        self.H = H
        self.syndrome = syndrome
        self.t = t
        self.max_iterations = 100
        
    def generate_random_permutation(self, n):
        P = Permutations(n).random_element().to_matrix()
        return P
    
    def get_weight(self, e):
        """
        Returns the weight of the error vector
        """
        weight = 0
        for ei in e:
            if ei != 0:
                weight += 1
        return weight
        
    def perform_attack(self):
        while True:
            r, n = self.H.dimensions()
            while True:
                P = self.generate_random_permutation(n)
                H_new = self.H * P
                # block_matrix - concatenating submatrices (rows first, then columns)
                # H_new | I_r (identity matrix placed to the right)
                H_new = block_matrix(1, (H_new, identity_matrix(r)))

                RREF = H_new.rref()
                U = RREF[:, n:n+r]
                V = RREF[:, r:n]
                W = RREF[:, 0:r]
                if W != identity_matrix(r):
                    break

            s = U * self.syndrome
            s = s.transpose().list()
            error = s +  [0] * (n - len(s)) 
    
            if self.get_weight(error) == self.t: # until weight = t
                return matrix(error) * P.T
            

                
    def check_property(self):
        nr_iter = 0
        max_iter = 10
        while True and nr_iter < max_iter:
            error = self.perform_attack()
#             print("-----")
#             print(self.H * error.transpose())
#             print("~~~~~~~~")
#             print(self.syndrome)
#             print("----")
            if self.H * error.transpose() == self.syndrome:
                print("correct: ", error, self.H * error.transpose())
                break
            if nr_iter == max_iter:
                print("maximum iterations")
                break
            nr_iter += 1
            

n = 8
# H: an r × n binary parity-check matrix
# matrix generated using goppa code in sage using jupyter notebook
H = Matrix(GF(2), matrix([[1, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 1, 1, 1, 0],
[0, 1, 1, 1, 0, 0, 1, 0],
[0, 1, 1, 1, 1, 1, 1, 1],
[0, 1, 0, 1, 1, 0, 1, 0],
[0, 0, 1, 1, 1, 1, 0, 0]])) # 6 x 8

# s: an r-bit long syndrome (column vector)
syndrome = Matrix(ZZ, matrix(6, 1, [0, 1, 1, 0, 0, 1]))  
# t: the weight of the error vector to be recovered
t_weight = 3

#Output: e: an n-bit binary row error vector s.t. H*e^T = s, with WEIGHT(e) = t

prange = PrangeISD(H, syndrome, t_weight)
# e, _, __ = prange.attack()
# print("OUTPUT:\n", H * e.transpose())

print(prange.generate_random_permutation(n))
prange.check_property()

# e = prange.perform_attack()

[0 1 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 0 1 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 0 1 0 0]
[1 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 1]
('correct: ', [1 0 1 0 0 0 0 1], [0]
[1]
[1]
[0]
[0]
[1])


In [6]:
def vector_t_errors(n,t):
        '''error vector with exactly t errors'''
        e = [1] * t + [0] * (n-t)
        shuffle(e)
        return list(e)
    
def generate_syndrome(n, t):
    s = vector_t_errors(n, t)
    syndrome = Matrix(GF(2), matrix(n, 1, s))
    return syndrome


class SyndromeMonitor:
    
    def check_syndrome(self, syndrome, dimension, t):
        '''property of syndrome not true'''
        if syndrome.nrows() != dimension or sum(syndrome.coefficients()) != t:
            return False
        return True
        
    def correct_syndrome(self, syndrome, dimension, t):

        # check if only ones and zeros
        for index, s in enumerate(syndrome):
            if s != 0 and s != 1:
                syndrome[index, 0] = GF(2)(1)
    

        # check length to be the same as matrix
        # check if exactly t ones
        if syndrome.nrows() != dimension or sum(syndrome.coefficients()) != t:
            syndrome = generate_syndrome(dimension, t)
            
        return syndrome


    def violation_handler(self, prop_name):
        print("Violation: Property", prop_name, ' - Syndrome incorrect!')

    def validation_handler(self, prop_name):
        print('Validation: Property', prop_name, ' - Syndrome is correct')
        
syndrome_monitor = SyndromeMonitor()

    
# when creating syndrome check the property (observation)
syndrome = Matrix(ZZ, matrix(6, 1, [0, 3, 1, 0, 0, 1]))  
print(syndrome)

print("--")
print(syndrome)

#verification
if not syndrome_monitor.check_syndrome(syndrome, 6, 3):
    syndrome_monitor.violation_handler('check_syndrome')
    syndrome = syndrome_monitor.correct_syndrome(syndrome, 6, 3)
    print("syndrome = ", syndrome)
else:
    syndrome.validation_handler('check_syndrome')

if not syndrome_monitor.check_syndrome(syndrome, 6, 3):
    syndrome_monitor.violation_handler('check_syndrome')
    syndrome = syndrome_monitor.correct_syndrome(syndrome, 6, 3)
else:
        syndrome_monitor.validation_handler('check_syndrome')

[0]
[3]
[1]
[0]
[0]
[1]
--
[0]
[3]
[1]
[0]
[0]
[1]
('Violation: Property', 'check_syndrome', ' - Syndrome incorrect!')
('syndrome = ', [0]
[1]
[1]
[0]
[0]
[1])
('Validation: Property', 'check_syndrome', ' - Syndrome is correct')
